# MindfulAI: QLoRA fine-tuning demo (Assignment H)
**Article theme:** “Let Mind Think And You Simply Be” — mindfulness, thought observation, self-awareness.

> Run in Google Colab with a GPU. This is a small educational experiment, not a clinical model. Do not claim training completed unless all cells finish.

## 1. Check GPU and install libraries

In [ ]:
!nvidia-smi
!pip -q install -U transformers datasets peft bitsandbytes accelerate

## 2. Load dataset

In [ ]:
import json, torch
from datasets import Dataset
from pathlib import Path

# If running this notebook in Colab, upload mindfulness_instructions.jsonl when prompted,
# or use the repository's raw GitHub URL after you create the repository.
from google.colab import files
uploaded = files.upload()  # select data/mindfulness_instructions.jsonl
filename = next(iter(uploaded))
records = [json.loads(line) for line in open(filename, encoding="utf-8") if line.strip()]
print("Examples:", len(records))
print(records[0])

## 3. Configure a small 4-bit model

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

use_4bit = torch.cuda.is_available()
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config if use_4bit else None,
    device_map="auto" if use_4bit else None,
    trust_remote_code=True,
)
if use_4bit:
    model = prepare_model_for_kbit_training(model)
print("GPU available:", torch.cuda.is_available())
print("Model loaded:", MODEL_NAME)

## 4. Baseline response before fine-tuning

In [ ]:
def make_prompt(instruction):
    messages = [
        {"role": "system", "content": "You are MindfulAI, a general wellbeing education assistant. Explain mindfulness in simple, non-judgmental language. You are not a therapist and do not diagnose. For persistent or severe distress, encourage professional support."},
        {"role": "user", "content": instruction},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

def generate_answer(current_model, question, max_new_tokens=120):
    prompt = make_prompt(question)
    inputs = tokenizer(prompt, return_tensors="pt")
    device = next(current_model.parameters()).device
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        out = current_model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False,
                                     pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

test_question = "How can I observe my thoughts without fighting them?"
print(generate_answer(model, test_question))

## 5. Prepare instruction examples and attach LoRA

In [ ]:
from peft import LoraConfig, get_peft_model
from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling

def format_example(ex):
    messages = [
        {"role": "system", "content": "You are MindfulAI, a general wellbeing education assistant. Explain mindfulness in simple, non-judgmental language. Do not diagnose or claim to provide therapy. Encourage professional support for persistent or severe distress."},
        {"role": "user", "content": ex["instruction"]},
        {"role": "assistant", "content": ex["response"]},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False)

texts = [format_example(x) for x in records]
raw_ds = Dataset.from_dict({"text": texts})

def tokenize_batch(batch):
    return tokenizer(batch["text"], truncation=True, max_length=384, padding="max_length")

tokenized = raw_ds.map(tokenize_batch, batched=True, remove_columns=["text"])
tokenized = tokenized.map(lambda x: {"labels": x["input_ids"].copy()})

# Qwen2.5 transformer projection layers
lora_config = LoraConfig(
    r=8, lora_alpha=16, lora_dropout=0.05, bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 6. Short training run

In [ ]:
training_args = TrainingArguments(
    output_dir="mindfulai-qlora-output",
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=1,
    save_strategy="epoch",
    fp16=torch.cuda.is_available(),
    report_to="none",
    optim="paged_adamw_8bit" if use_4bit else "adamw_torch",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized,
    data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False),
)
train_result = trainer.train()
print(train_result)

## 7. Test after fine-tuning and save adapter

In [ ]:
print("QUESTION:", test_question)
print("\\nAFTER FINE-TUNING:")
print(generate_answer(model, test_question))

ADAPTER_DIR = "mindfulai-lora-adapter"
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print("Saved adapter to", ADAPTER_DIR)

# Download adapter files as a zip for your records (optional)
import shutil
shutil.make_archive(ADAPTER_DIR, "zip", ADAPTER_DIR)
from google.colab import files
files.download(ADAPTER_DIR + ".zip")

## 8. Evaluation and limitations
Use the same 2–3 prompts before and after training. Compare relevance, clarity, safety, and whether the answer follows the requested style. A small dataset and one epoch do **not** establish clinical effectiveness or robust improvement. Record the actual output and any failures in the answer book.

Safety boundary: this is not therapy, diagnosis, or crisis support. Mindfulness may not suit everyone; users should stop if an exercise increases discomfort and seek professional help for persistent or severe distress.